# Prueba de estacionariedad (Augmented Dickey-Fuller)
H0: la serie tiene raíz unitaria (**no** estacionaria). Si **p < 0.05** se rechaza H0 → la serie es **estacionaria**.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
from statsmodels.tsa.stattools import adfuller

In [2]:
DATA_DIR = Path("datos")
TICKERS = ["NVDA", "MSFT", "GOOGL"]
NON_FEATURES = ["Date", "Target_Retorno_1d", "Target_Direccion"]

datasets = {t: pd.read_csv(DATA_DIR / f"{t}_dataset.csv", parse_dates=["Date"]) for t in TICKERS}
feature_cols = [c for c in datasets[TICKERS[0]].columns if c not in NON_FEATURES]
print(f"{len(feature_cols)} features:", feature_cols)

21 features: ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50', 'WMA_20', 'RSI_14', 'MACD', 'MACD_signal', 'BB_upper', 'BB_mid', 'BB_lower', 'Momentum_10', 'Stoch_K', 'Stoch_D', 'WilliamsR_14', 'SP500_ret', 'NASDAQ_ret', 'Gold_ret']


## Resultados (p-value por acción)

In [3]:
UMBRAL = 0.05

def pvalor_adf(serie):
    return adfuller(serie.dropna(), autolag="AIC")[1]

filas = []
for f in feature_cols:
    ps = {t: pvalor_adf(datasets[t][f]) for t in TICKERS}
    estacionaria = all(p < UMBRAL for p in ps.values())
    filas.append([f, ps["NVDA"], ps["MSFT"], ps["GOOGL"],
                  "ESTACIONARIA" if estacionaria else "NO estacionaria"])

resultados = pd.DataFrame(filas, columns=["feature", "p_NVDA", "p_MSFT", "p_GOOGL", "veredicto"])
display(resultados.round(4))

,feature,p_NVDA,p_MSFT,p_GOOGL,veredicto
0,Open,0.9986,0.9944,1.0000,NO estacionaria
1,High,0.9985,0.9933,1.0000,NO estacionaria
2,Low,0.9988,0.9955,1.0000,NO estacionaria
3,Close,0.9986,0.9949,1.0000,NO estacionaria
4,Volume,0.0004,0.0006,0.0001,ESTACIONARIA
5,SMA_20,1.0000,0.9942,0.9988,NO estacionaria
6,SMA_50,1.0000,0.9958,0.9988,NO estacionaria
7,WMA_20,1.0000,0.9954,0.9990,NO estacionaria
8,RSI_14,0.0000,0.0000,0.0000,ESTACIONARIA
9,MACD,0.0000,0.0000,0.0000,ESTACIONARIA


## Resumen

In [4]:
estacionarias = resultados.loc[resultados["veredicto"] == "ESTACIONARIA", "feature"].tolist()
no_estacionarias = resultados.loc[resultados["veredicto"] == "NO estacionaria", "feature"].tolist()
print("ESTACIONARIAS (p < 0.05):", estacionarias)
print("\nNO estacionarias:", no_estacionarias)

ESTACIONARIAS (p < 0.05): ['Volume', 'RSI_14', 'MACD', 'MACD_signal', 'Momentum_10', 'Stoch_K', 'Stoch_D', 'WilliamsR_14', 'SP500_ret', 'NASDAQ_ret', 'Gold_ret']

NO estacionarias: ['Open', 'High', 'Low', 'Close', 'SMA_20', 'SMA_50', 'WMA_20', 'BB_upper', 'BB_mid', 'BB_lower']


## Tratamiento de la variable no estacionaria con rezagos

In [5]:
from statsmodels.tsa.stattools import acf

VAR = "BB_upper"          # variable no estacionaria a tratar
S_LIST = [5, 21, 252]     # rezagos estacionales: semanal / mensual / anual

def transformaciones(df):
    """Transformaciones candidatas para volver estacionaria a VAR."""
    bb = df[VAR]
    out = {"nivel_original": bb, "dif_normal_lag1": bb.diff()}
    for s in S_LIST:
        out[f"dif_estacional_s{s}"] = bb - bb.shift(s)
    ancho = (df["BB_upper"] - df["BB_lower"]).replace(0, pd.NA)
    out["pctB_relativo"] = (df["Close"] - df["BB_lower"]) / ancho
    return out

nombres = list(transformaciones(datasets[TICKERS[0]]).keys())

filas = []
for nombre in nombres:
    ps = {t: pvalor_adf(transformaciones(datasets[t])[nombre]) for t in TICKERS}
    estacionaria = all(p < UMBRAL for p in ps.values())
    filas.append([nombre, ps["NVDA"], ps["MSFT"], ps["GOOGL"],
                  "ESTACIONARIA" if estacionaria else "NO estacionaria"])

resultados_bb = pd.DataFrame(filas, columns=["transformacion", "p_NVDA", "p_MSFT", "p_GOOGL", "veredicto"])
display(resultados_bb.round(4))

,transformacion,p_NVDA,p_MSFT,p_GOOGL,veredicto
0,nivel_original,0.9988,0.9968,1.0000,NO estacionaria
1,dif_normal_lag1,0.0000,0.0000,0.0000,ESTACIONARIA
2,dif_estacional_s5,0.0000,0.0000,0.0000,ESTACIONARIA
3,dif_estacional_s21,0.0000,0.0000,0.0000,ESTACIONARIA
4,dif_estacional_s252,0.2830,0.1312,0.3008,NO estacionaria
5,pctB_relativo,0.0000,0.0000,0.0000,ESTACIONARIA
